# 7. Evaluation

Measure retrieval, a model's accuracy on multiple-choice clinical questions, and
how faithful generated text is to its sources. OpenBTK ships harnesses, not
scores: these cells produce numbers for *stand-in* systems, to show how the
measurement works.

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

## Retrieval: recall@k, MRR, nDCG@k

In [2]:
from openbtk.eval.retrieval import RetrievalQuery, evaluate_retrieval

queries = [
    RetrievalQuery(
        query_id="q1", query="statin therapy", relevance={"c1": 1.0, "c2": 1.0}
    ),
    RetrievalQuery(query_id="q2", query="insulin dosing", relevance={"c3": 1.0}),
]
# A retriever is any function from a query to a ranked list of document ids.
ranked = {"statin therapy": ["c1", "c9", "c2"], "insulin dosing": ["c7", "c3"]}

report = evaluate_retrieval(lambda q: ranked[q], queries, ks=(1, 3))
print("recall@1 =", report.recall_at_k[1], " recall@3 =", report.recall_at_k[3])
print("MRR      =", report.mrr)
print("nDCG@3   =", round(report.ndcg_at_k[3], 3))

recall@1 = 0.25  recall@3 = 1.0
MRR      = 0.75
nDCG@3   = 0.775


Worked by hand: `q1` has two relevant documents and finds one at rank 1, so
recall@1 is 1/2; `q2` misses at rank 1, so 0; the average is 0.25.

## Clinical QA in MedQA / MedMCQA format

`evaluate_qa` scores any function from a prompt to a reply. The questions below are
invented and written in MedMCQA's file format. **No dataset and no benchmark score
ships with OpenBTK**; `read_medqa_jsonl` / `read_medmcqa_jsonl` read files you
download yourself.

In [3]:
import json
import pathlib
import tempfile

from openbtk.eval.qa import evaluate_qa, read_medmcqa_jsonl

rows = [
    {
        "id": "q1",
        "question": "Which vitamin deficiency causes scurvy?",
        "opa": "Vitamin A",
        "opb": "Vitamin C",
        "opc": "Vitamin D",
        "opd": "Vitamin K",
        "cop": 1,
        "subject_name": "Biochemistry",
    },
    {
        "id": "q2",
        "question": "Which organ produces insulin?",
        "opa": "Liver",
        "opb": "Kidney",
        "opc": "Pancreas",
        "opd": "Spleen",
        "cop": 2,
        "subject_name": "Physiology",
    },
    {
        "id": "q3",
        "question": "Which chamber pumps blood to the body?",
        "opa": "Right atrium",
        "opb": "Left ventricle",
        "opc": "Right ventricle",
        "opd": "Left atrium",
        "cop": 1,
        "subject_name": "Anatomy",
    },
]
path = pathlib.Path(tempfile.mkdtemp()) / "validation.jsonl"
path.write_text("\n".join(json.dumps(r) for r in rows), encoding="utf-8")

# A stand-in "model": right, wrong, and one reply that names no option at all.
replies = iter(["B", "The answer is A.", "I am not sure."])
qa = evaluate_qa(
    lambda prompt: next(replies), read_medmcqa_jsonl(path), keep_results=True
)

print(
    f"correct {qa.correct}/{qa.n}  unanswered {qa.unanswered}  accuracy {qa.accuracy:.2f}"
)
low, high = qa.accuracy_ci95
print(f"95% interval on three questions: {low:.2f} - {high:.2f}")
print({subject: (s.correct, s.n) for subject, s in sorted(qa.per_subject.items())})

correct 1/3  unanswered 1  accuracy 0.33
95% interval on three questions: 0.06 - 0.79
{'Anatomy': (0, 1), 'Biochemistry': (1, 1), 'Physiology': (0, 1)}


Two things to notice. The reply that names no option counts as **wrong** and is
also reported as `unanswered`, so a rambling model is visible. And on three
questions the 95% interval spans most of the range - read the interval, not just
the accuracy.

To score a real model: `evaluate_qa(LLMAnswerer(provider), items)`. The
`qa_manifest(...)` helper records which model, which file (by SHA-256) and when -
never the questions.

## Groundedness / faithfulness

In [4]:
from openbtk.eval.groundedness import (
    GroundedExample,
    LabelledExample,
    evaluate_detector,
    score_groundedness,
)

context = ["Assessment: type 2 diabetes mellitus, stable. Metformin continued."]
answers = [
    GroundedExample(
        example_id="a", answer="The patient has type 2 diabetes.", context=context
    ),
    GroundedExample(
        example_id="b", answer="The patient has a fractured femur.", context=context
    ),
]
faith = score_groundedness(answers)
print(
    f"faithfulness {faith.faithfulness:.2f} ({faith.supported_claims}/{faith.n_claims} claims supported)"
)

faithfulness 0.50 (1/2 claims supported)

That number comes from a **word-overlap heuristic**. Before trusting it, score
the *checker* against answers a human has judged:

In [5]:
judged = [
    LabelledExample(
        example_id="a", answer=answers[0].answer, context=context, grounded=True
    ),
    LabelledExample(
        example_id="b", answer=answers[1].answer, context=context, grounded=False
    ),
    # A paraphrase a human accepts but the heuristic cannot see - a false alarm:
    LabelledExample(
        example_id="c",
        answer="Glycaemia is controlled.",
        context=context,
        grounded=True,
    ),
]
det = evaluate_detector(judged)
print(f"precision {det.precision:.2f}  recall {det.recall:.2f}  f1 {det.f1:.2f}")

precision 0.50  recall 1.00  f1 0.67


The paraphrase `c` is a false positive, which is exactly the heuristic's limit.
Evaluate on your own labelled examples, and swap in a stronger `is_supported` if
this number matters.

## Manifests

Every evaluation can emit an `EvalManifest`: which component, which data
(SHA-256), when, and the counts - the same provenance discipline as a pipeline
run, and never any question, answer or note text.

In [6]:
from openbtk.eval.groundedness import groundedness_manifest
from openbtk.eval.qa import qa_manifest

manifest = qa_manifest(qa, source=path)
print(manifest.kind, "| input sha256:", manifest.input_digests[0].sha256[:12], "...")
assert "scurvy" not in manifest.model_dump_json()

print(groundedness_manifest(faith).kind)

qa | input sha256: c34e84f6676b ...
groundedness
